# Multi-Agent Research System

> Seven agents across eight graph nodes: a planner, three researchers running in
> parallel, a quality gate, an analyst, a synthesizer, a writer and a reviewer.

This notebook builds the same architecture as
[`projects/agentic-ai/ai-agents-project`](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/agentic-ai/ai-agents-project),
from scratch, in one file you can run top to bottom.

**It runs with or without an API key.** With no key, the agents talk to a small
deterministic stand-in so the orchestration is still watchable. Supply a free
Gemini key and the same graph runs for real.

What is worth your attention here is not the agents. It is three things the code
does not draw attention to:

1. Three agents write to the same state field at the same time, and one
   annotation is what makes that legal at all.
2. Two edges point backwards. A pipeline that cannot go backwards is a queue.
3. Every agent catches its own exceptions, so a run in which everything failed
   still completes and still prints something.

## 1. Install and configure

Only `langgraph` and `pydantic` are needed for the graph itself.
`langchain-google-genai` is used only if you supply a key.

In [ ]:
%pip install -q langgraph pydantic langchain-google-genai

In [ ]:
import os
from getpass import getpass

# OPTIONAL. Press Enter to skip and run against the deterministic stand-in.
key = getpass("Google API key (press Enter to run offline): ").strip()

PLACEHOLDERS = ("your-", "your_", "changeme", "xxx", "here")


def looks_real(value: str) -> bool:
    """A placeholder is a non-empty string, so a truthiness test is not enough.

    This exact bug shipped in the repo: .env.example fills every key with a
    placeholder, `not os.getenv(...)` therefore said a key was present, the
    offline fallback never fired, the real client raised, the exception was
    caught, and every search returned an empty list. The pipeline completed and
    printed a report built on nothing.
    """
    v = (value or "").strip()
    return len(v) >= 12 and not any(p in v.lower() for p in PLACEHOLDERS)


OFFLINE = not looks_real(key)
if not OFFLINE:
    os.environ["GOOGLE_API_KEY"] = key

print("OFFLINE (deterministic stand-in)" if OFFLINE else "LIVE (Gemini)")

## 2. State, and the annotation that matters most

`ResearchState` is an ordinary TypedDict with one unusual detail. Three
researchers run at the same time and all three write to `sources`, and a plain
field can hold only one value per step.

`Annotated[list, operator.add]` names a **reducer**: a function LangGraph uses to
combine concurrent writes instead of replacing them. For a list, `+` is
concatenation, so the three result sets join rather than overwrite.

In [ ]:
import operator
from typing import Annotated, TypedDict


class ResearchState(TypedDict, total=False):
    query: str
    sub_topics: list[str]
    research_plan: str

    # Written by THREE researchers concurrently. Without the reducer LangGraph
    # raises InvalidUpdateError rather than picking a winner. Section 8 measures
    # both this case and the sequential one, which does NOT raise.
    sources: Annotated[list[dict], operator.add]
    token_count: Annotated[int, operator.add]
    errors: Annotated[list[str], operator.add]
    pipeline_trace: Annotated[list[dict], operator.add]

    quality_score: float
    quality_passed: bool
    key_claims: list[dict]
    conflicts: list[str]
    synthesis: str
    current_draft: str
    review: dict
    revision_count: int
    final_report: str


print("Reducer fields:", [k for k, v in ResearchState.__annotations__.items()
                          if getattr(v, "__metadata__", None)])

## 3. The model, real or stand-in

Both paths expose the same two things the agents use: `.invoke()` for prose and
`.with_structured_output(Schema)` for typed replies.

The stand-in is a rule engine, not a model. It exists so the graph is watchable
without a key, and it **raises** on a prompt it does not recognise rather than
returning something plausible. A wrong reply that parses is far worse than a
crash, because every agent below catches exceptions and carries on.

In [ ]:
from pydantic import BaseModel, Field


class PlannerOutput(BaseModel):
    sub_topics: list[str] = Field(description="1-3 focused sub-topics", max_length=3)
    research_plan: str = Field(description="Brief strategy")


class Claim(BaseModel):
    claim: str
    evidence: str
    confidence: str = Field(description="high, medium or low")
    source_idx: int = 0


class AnalystOutput(BaseModel):
    claims: list[Claim]
    conflicts: list[str] = Field(default_factory=list)


class ReviewOutput(BaseModel):
    score: int = Field(ge=1, le=10)
    issues: list[str] = Field(default_factory=list)
    suggestions: list[str] = Field(default_factory=list)
    passed: bool


class OfflineFixtureError(RuntimeError):
    """Raised rather than answering a prompt we have no fixture for."""


class _Msg:
    def __init__(self, content, tokens):
        self.content = content
        # token_count() reads this. Reporting nothing would make the budget
        # guardrail read zero and silently stop protecting anything.
        self.usage_metadata = {"total_tokens": tokens}


class _Structured:
    def __init__(self, schema):
        self.schema = schema

    def invoke(self, prompt):
        name = self.schema.__name__
        if name == "PlannerOutput":
            topic = prompt.split("Query:")[-1].strip().rstrip("?") or "the topic"
            return self.schema(
                sub_topics=[f"Current state: {topic}",
                            f"Supporting evidence: {topic}",
                            f"Criticisms and limitations: {topic}"],
                research_plan="Split into state, evidence and limits; reconcile after.")
        if name == "AnalystOutput":
            # These deliberately DISAGREE. A conflicts list that is always empty
            # means the cross-referencing half of the pipeline never runs.
            return self.schema(
                claims=[
                    Claim(claim="Adoption grew sharply over two years.",
                          evidence="Two sources report year-on-year growth.",
                          confidence="high", source_idx=0),
                    Claim(claim="Unit costs are falling.",
                          evidence="Vendor pricing pages show reductions.",
                          confidence="medium", source_idx=2),
                    Claim(claim="Total spend has risen for most teams.",
                          evidence="A survey reports higher spend despite lower prices.",
                          confidence="medium", source_idx=1),
                    Claim(claim="No independent benchmark exists yet.",
                          evidence="Only a vendor blog claims this.",
                          confidence="low", source_idx=3),
                ],
                conflicts=["Unit costs falling against total spend rising. Both are "
                           "true; the disagreement is the finding."])
        if name == "ReviewOutput":
            # Fail the FIRST draft, so the revision loop actually executes.
            if "Revised after review" in prompt:
                return self.schema(score=8, issues=[], suggestions=[], passed=True)
            return self.schema(
                score=5,
                issues=["The low-confidence claim carries the same weight as the rest.",
                        "The cost contradiction is stated but never resolved."],
                suggestions=["Label the unverified claim.",
                             "Explain that unit cost and total spend can diverge."],
                passed=False)
        raise OfflineFixtureError(f"No offline fixture for {name}")


class OfflineLLM:
    def with_structured_output(self, schema):
        return _Structured(schema)

    def invoke(self, prompt):
        text = prompt if isinstance(prompt, str) else str(prompt)
        low = text.lower()
        if "synthes" in low:
            body = ("Growth is well supported. Cost is contested: unit prices fall "
                    "while total spend rises, which are not in conflict once you "
                    "separate price per call from number of calls. One claim rests "
                    "on a single vendor source and should be carried as unverified.")
        elif "report" in low or "draft" in low:
            revising = "Reviewer issues to fix" in text
            marker = ("\n\n> Revised after review: the unverified claim is now "
                      "labelled and the cost contradiction explained.\n" if revising else "")
            body = ("# Research Report\n" + marker +
                    "\n## Summary\n\nAdoption is growing and well evidenced. Costs "
                    "move in two directions at once.\n\n## Findings\n\n"
                    "1. Adoption grew sharply (high confidence).\n"
                    "2. Unit costs are falling while total spend rises (medium).\n\n"
                    "## Uncertainties\n\nOne claim has only a vendor source.\n")
        else:
            raise OfflineFixtureError(f"No offline fixture for prompt: {text[:80]}")
        return _Msg(body, max(1, len(text) // 4))


def get_llm():
    if OFFLINE:
        return OfflineLLM()
    from langchain_google_genai import ChatGoogleGenerativeAI
    return ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)


def token_count(response) -> int:
    """Zero, not a plausible guess, when the provider reports nothing.

    A zero is visibly wrong and gets noticed. A hardcoded 1500 looks like a
    measurement and quietly disables the budget guardrail.
    """
    usage = getattr(response, "usage_metadata", None) or {}
    return int(usage.get("total_tokens") or 0)


print("LLM ready:", type(get_llm()).__name__)

## 4. Search

Deterministic sources, so the notebook runs anywhere. Two things are on purpose:
the results **disagree with each other**, so the analyst has a real conflict to
find, and one source is weak, so the quality gate has something to reject.

In [ ]:
import hashlib

_TEMPLATES = [
    ("Adoption survey 2026", "https://example-research.org/survey-{s}",
     "Adoption grew year on year across every segment measured. Reported "
     "production reliability sat well below vendor demonstrations."),
    ("What {s} costs in practice", "https://engineering.example.com/cost-{s}",
     "Per-call pricing fell repeatedly. Total spend nonetheless rose for most "
     "teams, because volume grew faster than prices fell."),
    ("Vendor pricing update", "https://vendor.example.com/blog/{s}",
     "Costs continue to fall and will keep falling. No independent benchmark "
     "covers this area yet."),
    ("Notes on {s}", "https://someones-blog.example.net/{s}",
     "Short post. Mostly opinion, no sources cited."),
    ("Limitations of {s}", "https://example-university.edu/papers/{s}",
     "Three recurring failure modes; benchmarks overstate real-world results."),
]


def web_search(query: str, max_results: int = 5) -> list[dict]:
    slug = "-".join(w.lower() for w in query.split() if w.isalpha())[:40] or "topic"
    offset = int(hashlib.sha256(query.encode()).hexdigest(), 16) % len(_TEMPLATES)
    out = []
    for i in range(min(max_results, len(_TEMPLATES))):
        title, url, snippet = _TEMPLATES[(offset + i) % len(_TEMPLATES)]
        out.append({"title": title.format(s=slug), "url": url.format(s=slug),
                    "snippet": snippet})
    return out


print(f"{len(web_search('ai agents'))} sources for one query")

## 5. The seven agents

Each is a plain function taking the state and returning the fields it wrote.
Note that every one of them catches its own exceptions. That is deliberate in the
original, and it has a consequence worth holding on to: **the pipeline completes
even when nothing worked**.

In [ ]:
import time


def _trace(agent, tokens=0, summary=""):
    return [{"agent": agent, "tokens": tokens, "summary": summary}]


def planner(state):
    """1. Break the question into independently researchable sub-topics."""
    try:
        chain = get_llm().with_structured_output(PlannerOutput)
        result = chain.invoke(f"Break this into 1-3 sub-topics.\n\nQuery: {state['query']}")
        tokens = 400 if OFFLINE else 0
        return {"sub_topics": result.sub_topics[:3],
                "research_plan": result.research_plan,
                "token_count": tokens,
                "pipeline_trace": _trace("planner", tokens,
                                         f"{len(result.sub_topics)} sub-topics")}
    except Exception as e:
        return {"sub_topics": [state["query"]], "errors": [f"planner: {e}"],
                "pipeline_trace": _trace("planner", 0, "failed")}


def researcher(state):
    """2. One instance per sub-topic, all running at the same time."""
    query = state["query"]
    sources = web_search(query, max_results=5)
    return {"sources": sources,
            "pipeline_trace": _trace("researcher", 0, f"{len(sources)} sources")}


def quality_gate(state):
    """3. Arithmetic only. No model call, which is why it goes before the analyst."""
    sources = state.get("sources", [])
    if not sources:
        return {"quality_score": 0.0, "quality_passed": False,
                "pipeline_trace": _trace("quality_gate", 0, "no sources")}
    scored = []
    for s in sources:
        domain = 0.9 if ".edu" in s["url"] or ".org" in s["url"] else 0.5
        substance = min(len(s["snippet"]) / 200, 1.0)
        scored.append(0.6 * domain + 0.4 * substance)
    score = sum(sorted(scored, reverse=True)[:5]) / min(len(scored), 5)
    return {"quality_score": round(score, 3), "quality_passed": score >= 0.40,
            "pipeline_trace": _trace("quality_gate", 0,
                                     f"score {score:.2f} vs 0.40")}


def retry_researcher(state):
    """4. The same researcher on a broadened query, when the gate rejects."""
    sources = web_search(state["query"] + " comprehensive overview", max_results=5)
    return {"sources": sources,
            "pipeline_trace": _trace("retry_researcher", 0, f"{len(sources)} more")}


def analyst(state):
    """5. Extract claims AND record where sources contradict each other."""
    try:
        chain = get_llm().with_structured_output(AnalystOutput)
        sources_text = "\n".join(f"[{i}] {s['title']}: {s['snippet']}"
                                 for i, s in enumerate(state.get("sources", [])[:10]))
        result = chain.invoke(f"Extract claims with evidence.\n\n{sources_text}")
        tokens = 1500 if OFFLINE else 0
        return {"key_claims": [c.model_dump() for c in result.claims],
                "conflicts": result.conflicts, "token_count": tokens,
                "pipeline_trace": _trace("analyst", tokens,
                                         f"{len(result.claims)} claims, "
                                         f"{len(result.conflicts)} conflicts")}
    except Exception as e:
        return {"key_claims": [], "errors": [f"analyst: {e}"],
                "pipeline_trace": _trace("analyst", 0, "failed")}


def synthesizer(state):
    """6. Reconcile the conflicts rather than picking a side."""
    try:
        llm = get_llm()
        claims = "\n".join(f"- [{c['confidence']}] {c['claim']}"
                            for c in state.get("key_claims", []))
        conflicts = "\n".join(state.get("conflicts", []))
        resp = llm.invoke(f"Write a synthesis.\n\nClaims:\n{claims}\n\n"
                          f"Known conflicts:\n{conflicts}")
        tokens = token_count(resp)
        return {"synthesis": resp.content, "token_count": tokens,
                "pipeline_trace": _trace("synthesizer", tokens, "reconciled")}
    except Exception as e:
        return {"synthesis": "", "errors": [f"synthesizer: {e}"],
                "pipeline_trace": _trace("synthesizer", 0, "failed")}


def writer(state):
    """7. First draft, or a revision using the reviewer's specific feedback."""
    try:
        llm = get_llm()
        n = state.get("revision_count", 0)
        if n == 0:
            prompt = (f"Write a research report.\n\nSynthesis:\n"
                      f"{state.get('synthesis', '')}")
        else:
            review = state.get("review", {})
            prompt = ("Revise this report.\n\nCurrent draft:\n"
                      f"{state.get('current_draft', '')}\n\n"
                      "Reviewer issues to fix:\n"
                      + "\n".join(f"- {i}" for i in review.get("issues", [])))
        resp = llm.invoke(prompt)
        tokens = token_count(resp)
        return {"current_draft": resp.content, "revision_count": n + 1,
                "token_count": tokens,
                "pipeline_trace": _trace("writer", tokens, f"draft {n + 1}")}
    except Exception as e:
        return {"current_draft": "", "errors": [f"writer: {e}"],
                "pipeline_trace": _trace("writer", 0, "failed")}


def reviewer(state):
    """8. Score the draft. Below 7 sends it back to the writer."""
    try:
        chain = get_llm().with_structured_output(ReviewOutput)
        result = chain.invoke("Score this report 1-10.\n\nReport to review:\n"
                              + state.get("current_draft", ""))
        tokens = 800 if OFFLINE else 0
        review = result.model_dump()
        out = {"review": review, "token_count": tokens,
               "pipeline_trace": _trace("reviewer", tokens,
                                        f"score {result.score}")}
        if result.passed or state.get("revision_count", 0) >= 2:
            out["final_report"] = state.get("current_draft", "")
        return out
    except Exception as e:
        return {"review": {"score": 0, "passed": False},
                "final_report": state.get("current_draft", ""),
                "errors": [f"reviewer: {e}"],
                "pipeline_trace": _trace("reviewer", 0, "failed")}


print("7 agent functions defined")

## 6. Wiring, and the two edges that go backwards

`Send()` is the fan-out: one researcher instance per sub-topic, dispatched
together. The two conditional edges are what make this a graph rather than a
queue: the quality gate can send research back, and the reviewer can send the
draft back.

In [ ]:
from langgraph.graph import END, START, StateGraph
from langgraph.types import Send


def route_to_researchers(state):
    """Fan-out: one researcher per sub-topic, all at once."""
    return [Send("researcher", {"query": t}) for t in state.get("sub_topics", [])]


def route_after_quality(state):
    return "analyst" if state.get("quality_passed") else "retry_researcher"


def route_after_review(state):
    review = state.get("review", {})
    if review.get("passed") or state.get("revision_count", 0) >= 2:
        return END
    return "writer"          # <- backwards


def build_graph():
    g = StateGraph(ResearchState)
    for name, fn in (("planner", planner), ("researcher", researcher),
                     ("quality_gate", quality_gate),
                     ("retry_researcher", retry_researcher), ("analyst", analyst),
                     ("synthesizer", synthesizer), ("writer", writer),
                     ("reviewer", reviewer)):
        g.add_node(name, fn)

    g.add_edge(START, "planner")
    g.add_conditional_edges("planner", route_to_researchers, ["researcher"])
    g.add_edge("researcher", "quality_gate")
    g.add_conditional_edges("quality_gate", route_after_quality,
                            ["analyst", "retry_researcher"])
    g.add_edge("retry_researcher", "analyst")
    g.add_edge("analyst", "synthesizer")
    g.add_edge("synthesizer", "writer")
    g.add_edge("writer", "reviewer")
    g.add_conditional_edges("reviewer", route_after_review, ["writer", END])
    return g.compile()


app = build_graph()
print("8 nodes compiled")

## 7. Run it

In [ ]:
result = app.invoke({"query": "What are the latest trends in AI agents?"})

route = [t["agent"] for t in result.get("pipeline_trace", [])]
print("ROUTE")
print("  " + " -> ".join(route))
print()
print(f"  sub-topics       {len(result.get('sub_topics', []))}")
print(f"  researchers      {route.count('researcher')} in parallel")
print(f"  sources          {len(result.get('sources', []))}")
print(f"  quality score    {result.get('quality_score')} (threshold 0.40)")
print(f"  claims           {len(result.get('key_claims', []))}")
print(f"  conflicts        {len(result.get('conflicts', []))}")
print(f"  drafts written   {result.get('revision_count')}")
print(f"  reviewer score   {result.get('review', {}).get('score')}")
print(f"  tokens           {result.get('token_count')}")
print(f"  errors           {result.get('errors', [])}")

Look at the route before anything else.

`researcher` appears three times because the planner produced three sub-topics
and each became a `Send()`. If `writer` and `reviewer` both appear twice, the
reviewer rejected the first draft and the graph went backwards.

`errors` should be an empty list. It is the only reliable signal that the run
worked, because every agent catches its own exceptions and the pipeline
completes either way.

## 8. What happens without the reducer

Worth testing rather than assuming. The behaviour depends on **when** the two
writes happen, and only one of the two cases is dangerous.

In [ ]:
from langgraph.errors import InvalidUpdateError


class BrokenState(TypedDict, total=False):
    """Identical, except `sources` has no reducer."""
    query: str
    sub_topics: list[str]
    sources: list[dict]                       # <- no reducer
    token_count: Annotated[int, operator.add]
    errors: Annotated[list[str], operator.add]
    pipeline_trace: Annotated[list[dict], operator.add]
    quality_score: float
    quality_passed: bool
    key_claims: list[dict]
    conflicts: list[str]
    synthesis: str
    current_draft: str
    review: dict
    revision_count: int
    final_report: str


def build_broken():
    g = StateGraph(BrokenState)
    for name, fn in (("planner", planner), ("researcher", researcher),
                     ("quality_gate", quality_gate),
                     ("retry_researcher", retry_researcher), ("analyst", analyst),
                     ("synthesizer", synthesizer), ("writer", writer),
                     ("reviewer", reviewer)):
        g.add_node(name, fn)
    g.add_edge(START, "planner")
    g.add_conditional_edges("planner", route_to_researchers, ["researcher"])
    g.add_edge("researcher", "quality_gate")
    g.add_conditional_edges("quality_gate", route_after_quality,
                            ["analyst", "retry_researcher"])
    g.add_edge("retry_researcher", "analyst")
    g.add_edge("analyst", "synthesizer")
    g.add_edge("synthesizer", "writer")
    g.add_edge("writer", "reviewer")
    g.add_conditional_edges("reviewer", route_after_review, ["writer", END])
    return g.compile()


try:
    build_broken().invoke({"query": "What are the latest trends in AI agents?"})
    print("no error raised")
except InvalidUpdateError as e:
    print("InvalidUpdateError:")
    print(" ", str(e).splitlines()[0])

LangGraph refuses. Three researchers try to write `sources` in the same
superstep, the channel can hold only one value, and you get an error that names
the offending key.

That is the good case, and it means you cannot ship a fan-out with a missing
reducer by accident. Do not take the lesson to be "the framework will catch it",
though, because the protection only covers **concurrent** writes.

In [ ]:
# Two nodes writing the same un-annotated key, one after the other.
class Sequential(TypedDict, total=False):
    items: list                                   # no reducer
    log: Annotated[list, operator.add]


def first(state):
    return {"items": ["from-first"], "log": ["first"]}


def second(state):
    return {"items": ["from-second"], "log": ["second"]}


g = StateGraph(Sequential)
g.add_node("first", first)
g.add_node("second", second)
g.add_edge(START, "first")
g.add_edge("first", "second")
g.add_edge("second", END)

out = g.compile().invoke({})
print("items:", out["items"])
print("log:  ", out["log"])
print()
print("No error. The second node REPLACED the first node's value, and the only")
print("evidence that anything was lost is a list you were not counting.")

There is the real trap. Sequential writes to a field with no reducer overwrite
silently, and there is no exception, no warning and no log line.

So the rule is not "add reducers to parallel fields". It is: **for every state
field, know how many nodes write it and in what order.** The concurrent case
fails loudly. The sequential one does not fail at all.

## What to take away

- **A reducer is not an optimisation.** Without it a fan-out will not run at
  all: LangGraph raises and names the key. The silent case is sequential writes,
  which is the one to watch for.
- **Put the cheap gate before the expensive agents.** The quality gate makes no
  model call at all; it is arithmetic placed where it can prevent spend.
- **Let the graph go backwards.** The reviewer sending a draft back is the only
  part of this pipeline that can correct a mistake.
- **"It completed" is not "it worked."** Every agent swallows its own
  exceptions. Assert on what was produced, not on the absence of a crash.
- **A placeholder key is a truthy string.** That one line of carelessness made
  the real project return zero sources while printing a full report.
- **Check which failures are loud before you rely on it.** I assumed the missing
  reducer lost data quietly; measuring showed the opposite.

The full project adds guardrails (token budget, rate limiting, PII scrubbing,
prompt-injection sanitising), a SQLite cache, a Streamlit UI, and an evaluation
comparing this against a single-agent baseline:

https://github.com/genieincodebottle/aiml-companion/tree/main/projects/agentic-ai/ai-agents-project

`python run.py demo` there runs the real graph with no API key.